# 🍃 Leaf Disease & Quality AI: Training & Edge AI Export

### โครงการนวัตกรรม AIoT และการเกษตรแม่นยำ (CMU AIoT & Scirbru Workshop)



[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tsanaphy2023/aiot-workshop/blob/main/leaf_ai_system/Leaf_AI_Training_Colab.ipynb)

[![GitHub Repository](https://img.shields.io/badge/GitHub-aiot--workshop-blue?logo=github)](https://github.com/Tsanaphy2023/aiot-workshop)



สมุดบันทึกนี้ออกแบบสำหรับการฝึกสอน (Train) โมเดลจำแนกโรคพืชและคุณภาพผลผลิต 6 ชนิด (**ข้าวโพด, กาแฟ, มันฝรั่ง, ส้ม, มะม่วง, กล้วย**) ด้วยโมเดล **MobileNetV2 / ResNet18 / LeafNet** โดยใช้การเร่งความเร็วผ่าน **Free GPU (Tesla T4)** บน Google Colab


## 1. ตรวจสอบการเปิดใช้งาน GPU บน Google Colab

> *หมายเหตุ: ไปที่เมนู `Runtime` -> `Change runtime type` แล้วเลือก `T4 GPU`*


In [ ]:
!nvidia-smi



import torch

print(f"PyTorch Version: {torch.__version__}")

print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():

    print(f"GPU Device: {torch.cuda.get_device_name(0)}")


## 2. ดึงโค้ดโปรเจกต์จาก GitHub และเตรียมชุดข้อมูล

ดาวน์โหลดโค้ดระบบ `leaf_ai_system` และติดตั้งไลบรารีที่จำเป็น


In [ ]:
import os



# รีเซ็ตไดเรกทอรีการทำงานกลับสู่ /content เสมอ เพื่อป้องกันโฟลเดอร์ซ้อนกันเมื่อรันเซลล์ซ้ำ

%cd /content



# โคลนโปรเจกต์จาก GitHub หากยังไม่มี หรือดึงอัปเดตล่าสุด

if not os.path.exists('/content/aiot-workshop'):

    !git clone https://github.com/Tsanaphy2023/aiot-workshop.git

else:

    print('🔄 กำลังดึงโค้ดเวอร์ชันล่าสุดจาก GitHub...')

    !cd /content/aiot-workshop && git pull origin main



%cd /content/aiot-workshop/leaf_ai_system

!pip install -q torchvision scikit-learn seaborn matplotlib pandas onnx onnxscript tqdm tabulate


## 3. แตกไฟล์ชุดข้อมูล Leaf Workshop (หากรันจากไฟล์ .rar)


In [ ]:
import os
import glob
from pathlib import Path

# ติดตั้ง unrar หากจำเป็น
!sudo apt-get install -y unrar > /dev/null 2>&1

dataset_dir = Path("/content/aiot-workshop/leaf workshop")
alt_dataset_dir = Path("/content/leaf workshop")

def check_dataset_ready():
    for d in [dataset_dir, alt_dataset_dir, Path("../leaf workshop")]:
        if d.exists() and any(d.iterdir()):
            crops = [p.name for p in d.iterdir() if p.is_dir()]
            if crops:
                print(f"✅ ตรวจพบชุดข้อมูลเรียบร้อยแล้วที่: {d.resolve()}")
                print(f"🌱 หมวดหมู่พืชที่พร้อมใช้งาน: {crops}")
                return True
    return False

if not check_dataset_ready():
    print("🔍 กำลังค้นหาไฟล์ชุดข้อมูลบีบอัด (.rar หรือ .zip)...")
    archive_candidates = (
        glob.glob("/content/*leaf*workshop*.rar") +
        glob.glob("/content/*leaf*workshop*.zip") +
        glob.glob("/content/aiot-workshop/*leaf*workshop*.rar") +
        glob.glob("/content/aiot-workshop/*leaf*workshop*.zip") +
        glob.glob("../*leaf*workshop*.rar") +
        glob.glob("../*leaf*workshop*.zip")
    )
    if archive_candidates:
        archive_file = archive_candidates[0]
        print(f"📦 กำลังแตกไฟล์ชุดข้อมูลจาก: {archive_file}...")
        dest_dir = "/content/aiot-workshop"
        if archive_file.endswith(".rar"):
            !unrar x -inul "{archive_file}" "{dest_dir}/"
        else:
            !unzip -q -o "{archive_file}" -d "{dest_dir}/"
        print("✅ แตกไฟล์ชุดข้อมูลเสร็จสิ้น!")
        check_dataset_ready()
    else:
        print("⚠️ ยังไม่พบโฟลเดอร์ leaf workshop หรือไฟล์ leaf workshop.rar บน Colab VM")
        print("💡 โปรดเลือกวิธีนำเข้าชุดข้อมูลวิธีใดวิธีหนึ่งด้านล่างนี้:")
        print("---------------------------------------------------------------------------------")
        print("👉 วิธีที่ 1 (แนะนำ - สะดวกที่สุด): เชื่อมต่อ Google Drive ที่มีโฟลเดอร์ leaf workshop")
        print("   from google.colab import drive")
        print("   drive.mount('/content/drive')")
        print("   !cp -r '/content/drive/MyDrive/leaf workshop' /content/aiot-workshop/")
        print("---------------------------------------------------------------------------------")
        print("👉 วิธีที่ 2: อัปโหลดไฟล์ leaf workshop.rar ขึ้นมาที่ Colab")
        print("   from google.colab import files")
        print("   files.upload()  # เลือกไฟล์ leaf workshop.rar")
        print("   !unrar x -inul '/content/leaf workshop.rar' /content/aiot-workshop/")
        print("---------------------------------------------------------------------------------")
else:
    print("🚀 ชุดข้อมูลพร้อมสำหรับการฝึกสอนโมเดลแล้ว!")


## 4. ขั้นตอนที่ 1: สั่งแบ่งข้อมูล Train (70%) / Val (15%) / Test (15%)

เลือกพืชที่ต้องการ: `corn`, `coffee`, `potato`, `banana`, `mango`, `orange`, หรือ `all`


In [ ]:
TARGET_CROP = "corn"  # ตัวเลือก: corn, coffee, potato, banana, mango, orange, all



!python scripts/01_split_dataset.py --crop {TARGET_CROP} --train_ratio 0.70 --val_ratio 0.15 --test_ratio 0.15


## 5. ขั้นตอนที่ 2: เริ่มต้นการฝึกสอนโมเดล (Training on GPU)

เลือกโมเดล Backbone: `mobilenet_v2` (แนะนำสำหรับ Edge AI), `resnet18`, หรือ `leaf_net` (Custom CNN)


In [ ]:
BACKBONE = "mobilenet_v2"  # mobilenet_v2, resnet18, leaf_net
EPOCHS = 15
BATCH_SIZE = 32
LEARNING_RATE = 0.001

!python scripts/02_train.py --crop {TARGET_CROP} --backbone {BACKBONE} --epochs {EPOCHS} --batch_size {BATCH_SIZE} --lr {LEARNING_RATE}


## 6. ดูกราฟประวัติการฝึกสอน (Loss & Accuracy Curves)


In [ ]:
from IPython.display import Image, display

curve_img = f"outputs/reports/{TARGET_CROP}_{BACKBONE}_curves.png"

if os.path.exists(curve_img):

    display(Image(curve_img))

else:

    print("Curve image not found.")


## 7. ขั้นตอนที่ 3: ประเมินผลโมเดลบน Test Set & Confusion Matrix


In [ ]:
checkpoint_file = f"outputs/checkpoints/{TARGET_CROP}_{BACKBONE}_best.pth"



!python scripts/03_evaluate.py --checkpoint {checkpoint_file}



cm_img = f"outputs/reports/{TARGET_CROP}_{BACKBONE}_confusion_matrix.png"

if os.path.exists(cm_img):

    display(Image(cm_img))


## 8. ขั้นตอนที่ 4: ทดสอบทำนายภาพจริง (Inference Playground)


In [ ]:
import glob

from pathlib import Path

import pandas as pd



# สุ่มภาพตัวอย่างจากชุดทดสอบมาทดลองรัน Inference

test_csv_path = Path(f'data/splits/{TARGET_CROP}/test.csv')

sample_img = None



if test_csv_path.exists():

    df_test = pd.read_csv(test_csv_path)

    if len(df_test) > 0:

        from utils.dataset import LeafDataset

        ds = LeafDataset(test_csv_path)

        sample_img = str(ds._resolve_image_path(df_test.iloc[0]))



if not sample_img or not Path(sample_img).exists():

    for ext in ['*.jpg', '*.JPG', '*.jpeg', '*.JPEG', '*.png', '*.PNG']:

        found = (

            glob.glob(f'../leaf workshop/{TARGET_CROP}*/*/{ext}') +

            glob.glob(f'/content/leaf workshop/{TARGET_CROP}*/*/{ext}') +

            glob.glob(f'/content/aiot-workshop/leaf workshop/{TARGET_CROP}*/*/{ext}')

        )

        if found:

            sample_img = found[0]

            break



if sample_img and Path(sample_img).exists():

    print(f'Testing Image: {sample_img}')

    display(Image(sample_img, width=250))

    !python scripts/04_predict.py --checkpoint {checkpoint_file} --image "{sample_img}"

else:

    print('No sample image found.')


## 9. ขั้นตอนที่ 5: ส่งออกโมเดลเป็น ONNX และ TorchScript สำหรับบอร์ด Edge AI


In [ ]:
!python models/export.py --checkpoint {checkpoint_file}



# ดาวน์โหลดไฟล์โมเดลลงเครื่องคอมพิวเตอร์ของคุณ

from google.colab import files

import glob



for f in glob.glob(f"outputs/exported/{TARGET_CROP}_{BACKBONE}_best*"):

    print(f"Ready to download: {f}")

    # files.download(f)  # ปลด comment เพื่อดาวน์โหลดอัตโนมัติ


## 10. ขั้นตอนที่ 6: การประยุกต์ใช้ Object Detection และ Bounding Box (YOLO / Roboflow Format)

การตีกรอบ **Bounding Box** ช่วยให้โมเดล AI ระบุพิกัดตำแหน่งรอยโรค (Lesion Localization) และคัดแยกผลผลิตเป็นรายผลบนสายพานลำเลียงได้ โดยแปลงพิกัดเป็นมาตรฐาน **YOLO Normalized Format**: 


In [ ]:
# ทดสอบรัน Object Detection ตีกรอบ Bounding Box บนภาพใบพืช
import glob
from IPython.display import Image, display

# ค้นหาภาพตัวอย่างที่มีอยู่ในชุดข้อมูลโดยอัตโนมัติ
sample_files = glob.glob("../leaf workshop/*/*/*.jpg")
target_sample = sample_files[0] if sample_files else "../leaf workshop/corn_dataset/Blight/Corn_Blight (1).jpg"
print(f"🎯 Target Test Image: {target_sample}")

!python scripts/05_object_detection.py --image "{target_sample}" --crop auto --conf_threshold 0.35 --iou_threshold 0.45

# แสดงภาพผลลัพธ์ที่ตีกรอบ Bounding Box พร้อมค่าความมั่นใจ (%)
det_imgs = sorted(glob.glob("outputs/detections/*.jpg"))
if det_imgs:
    print(f"\n🖼️ Annotated Detection Image: {det_imgs[-1]}")
    display(Image(det_imgs[-1], width=550))
else:
    print("No annotated detection output found.")



---
# ภาคพิเศษ: การฝึกสอนโมเดลตรวจจับวัตถุระดับสถานะศิลป์ (YOLO11 & YOLOv8)

ในส่วนนี้เป็นการนำสถาปัตยกรรม **YOLO11 (Ultralytics)** มาใช้ตรวจจับวัตถุขนาดเล็กและรอยโรคพืชในระดับ Bounding Box รองรับทั้งงาน **พยาธิวิทยาพืช (Plant Pathology)** และงาน **เฝ้าระวังลูกน้ำยุงระบาดวิทยา (Vector Surveillance)** ผ่านคลาวด์ Roboflow Universe


## 11. ติดตั้งไลบรารี YOLO11 และตรวจสอบการเชื่อมต่อ GPU

ติดตั้งไลบรารี `ultralytics`, `supervision` และ `roboflow` พร้อมตรวจสอบสถานะชิปเร่งความเร็ว Tesla T4 GPU


In [ ]:
# ติดตั้งไลบรารีเวอร์ชันล่าสุดสำหรับการประมวลผล Computer Vision
%pip install -q "ultralytics==8.3.40" supervision roboflow

import ultralytics
import torch

# ตรวจสอบสภาพแวดล้อมระบบและชิป GPU
ultralytics.checks()
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")


## 12. กรณีศึกษาที่ 1: ตรวจจับโรคใบกาแฟด้วย YOLO11 (Smart Select Coffee Pathology)

ดาวน์โหลดชุดข้อมูลกาแฟ 3 คลาส (`leaf_rust`, `phoma`, `healthy`) ที่ผ่านการสกัดรอยโรคด้วย **Smart Select Bounding Box** จาก Roboflow มาฝึกสอนโมเดล YOLO11 Nano


In [ ]:
# ดาวน์โหลดชุดข้อมูลกาแฟ Smart Select จาก Roboflow
from roboflow import Roboflow

rf = Roboflow(api_key="tdetHe9WG2CTItfj0VHt")
project_coffee = rf.workspace("durian-nodisease").project("aiot_workshop2026")

# ดาวน์โหลดเวอร์ชันล่าสุดสำหรับ YOLO11
dataset_coffee = project_coffee.version(1).download("yolov11")
print(f"✅ ชุดข้อมูลใบกาแฟพร้อมใช้งานที่: {dataset_coffee.location}")


In [ ]:
# สั่งฝึกสอนโมเดล YOLO11 Nano บนชุดข้อมูลโรคใบกาแฟ
from ultralytics import YOLO

# โหลด Pre-trained Weights YOLO11 Nano
model_coffee = YOLO("yolo11n.pt")

# เริ่มกระบวนการฝึกสอน
results_coffee = model_coffee.train(
    data=f"{dataset_coffee.location}/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    plots=True,
    name="coffee_yolo11_run"
)


In [ ]:
# ดูกราฟประวัติการฝึกสอนและค่าความแม่นยำ mAP50 / mAP50-95 ของโรคใบกาแฟ
from IPython.display import Image, display
import os

results_img = "runs/detect/coffee_yolo11_run/results.png"
if os.path.exists(results_img):
    print("📊 กราฟผลการฝึกสอน YOLO11 (Box Loss, Class Loss, mAP50, Precision, Recall):")
    display(Image(results_img, width=800))
else:
    print("Results image not found yet.")


## 13. กรณีศึกษาที่ 2: โมเดลจำแนกชนิดลูกน้ำยุงระบาดวิทยา (Mosquito Larvae Surveillance)

ตรวจจับและคัดแยกชนิดพันธุ์ลูกน้ำยุงลายและยุงรำคาญ 3 คลาสสำคัญทางการแพทย์และระบาดวิทยา:
1. 🔴 **`Aedes_aegypti`** (ลูกน้ำยุงลายบ้าน — พาหะนำโรคไข้เลือดออก ชิคุนกุนยา ซิกา)
2. 🟢 **`Aedes_albopictus`** (ลูกน้ำยุงลายสวน)
3. 🟣 **`Culex_quinque`** (*Culex quinquefasciatus* — ลูกน้ำยุงรำคาญ พาหะโรคเท้าช้างและไข้สมองอักเสบ)

ชุดข้อมูล Version 3 ประกอบด้วยภาพ **2,730 ภาพ** ผ่านการ Preprocessing & Augmentation ระดับสากลเรียบร้อยแล้ว


In [ ]:
# ดาวน์โหลดชุดข้อมูลลูกน้ำยุงลาย (Mosquito Larvae v3) ในฟอร์แมต yolo26
from roboflow import Roboflow

rf = Roboflow(api_key="tdetHe9WG2CTItfj0VHt")
project_mosquito = rf.workspace("durian-nodisease").project("mosquito_larvae-ivnri")

# ดาวน์โหลด Version 3 ฟอร์แมต yolo26
version_mosquito = project_mosquito.version(3)
dataset_mosquito = version_mosquito.download("yolo26")

print(f"✅ ชุดข้อมูลลูกน้ำยุงพร้อมใช้งานที่: {dataset_mosquito.location}")


In [ ]:
# สั่งฝึกสอนโมเดลตรวจจับลูกน้ำยุงลายด้วย YOLO11
from ultralytics import YOLO

model_mosquito = YOLO("yolo11n.pt")

results_mosquito = model_mosquito.train(
    data=f"{dataset_mosquito.location}/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    plots=True,
    name="mosquito_yolo11_run"
)


In [ ]:
# ประเมินผลความแม่นยำ mAP และ Confusion Matrix ของโมเดลตรวจจับลูกน้ำยุง
metrics_mosquito = model_mosquito.val()
print(f"🎯 ค่าความแม่นยำเฉลี่ย mAP50: {metrics_mosquito.box.map50 * 100:.2f}%")
print(f"🎯 ค่าความแม่นยำรวม mAP50-95: {metrics_mosquito.box.map * 100:.2f}%")

cm_mosquito_img = "runs/detect/mosquito_yolo11_run/confusion_matrix.png"
if os.path.exists(cm_mosquito_img):
    display(Image(cm_mosquito_img, width=650))


## 14. ส่งออกโมเดลสู่มาตรฐานสากล (ONNX & TensorFlow Lite INT8) สำหรับ Edge AI & IoT

แปลงโมเดลให้อยู่ในมาตรฐานเปิด **ONNX** สำหรับรันบน Raspberry Pi / Jetson และ **TensorFlow Lite INT8** สำหรับติดตั้งบนกล้องสมาร์ทฟาร์ม ESP32-S3


In [ ]:
# ส่งออกโมเดลโรคพืช (Coffee) สู่ ONNX และ TFLite
best_coffee_weights = "runs/detect/coffee_yolo11_run/weights/best.pt"
if os.path.exists(best_coffee_weights):
    m_c = YOLO(best_coffee_weights)
    m_c.export(format="onnx", imgsz=640)
    m_c.export(format="tflite", imgsz=320, int8=True)
    print("✅ ส่งออกโมเดลโรคใบกาแฟสู่ ONNX และ TFLite INT8 เรียบร้อย!")

# ส่งออกโมเดลลูกน้ำยุง (Mosquito) สู่ ONNX และ TFLite
best_mosquito_weights = "runs/detect/mosquito_yolo11_run/weights/best.pt"
if os.path.exists(best_mosquito_weights):
    m_m = YOLO(best_mosquito_weights)
    m_m.export(format="onnx", imgsz=640)
    m_m.export(format="tflite", imgsz=320, int8=True)
    print("✅ ส่งออกโมเดลลูกน้ำยุงสู่ ONNX และ TFLite INT8 เรียบร้อย!")


In [ ]:
# สรุปขนาดไฟล์โมเดลพร้อมดาวน์โหลดลงคอมพิวเตอร์
import glob

print("📦 รายการไฟล์โมเดล Edge AI พร้อมนำไปติดตั้งภาคสนาม:")
for pattern in ["runs/detect/*_run/weights/best.onnx", "runs/detect/*_run/weights/*_int8.tflite"]:
    for mf in glob.glob(pattern):
        sz_mb = os.path.getsize(mf) / (1024 * 1024)
        print(f"  - {mf} (ขนาด: {sz_mb:.2f} MB)")

# ปลด comment บรรทัดล่างสุดเพื่อดาวน์โหลดไฟล์ลงเครื่องคอมพิวเตอร์ผ่านเบราว์เซอร์:
# from google.colab import files
# files.download('runs/detect/coffee_yolo11_run/weights/best.onnx')
